# Couche Silver KBO Open Data : `entreprise` → `entreprise_silver`

## 0. Contexte : bronze vs silver

La couche **bronze** (`entreprise`) reste sous forme **brute** :

- des codes non traduits (`Status="AC"`, `TypeOfAddress="REGO"`...) ;
- des tableaux indexés `0/1/2/3` sans clé porteuse de sens ;
- des champs dupliqués par langue (`CountryNL`/`CountryFR`, `MunicipalityNL`/`MunicipalityFR`...) ;
- des activités qui réapparaissent en double sous plusieurs versions NACE (2003, 2008, 2025) pour la même réalité.

La couche **silver** de transformation géres tout ça

### Ce que « gérer tout ça » veut dire concrètement

Le bronze répond à *« où sont les données ? »*. Le silver répond à *« que
veulent-elles dire ? »*. Quatre transformations, et une règle transversale.

| # | Problème du bronze | Réponse du silver |
|---|---|---|
| 1 | `Status="AC"`, `JuridicalForm="014"` | traduction en français via `kbo_code` |
| 2 | `denominations[0]`, `denominations[1]`... | dictionnaire clé par **type traduit** |
| 3 | `MunicipalityNL` **et** `MunicipalityFR` | une seule langue, colonne unique |
| 4 | la même activité répétée en NACE 2003 / 2008 / 2025 | dédoublonnage, version la plus récente |

**Règle transversale : le silver ne conserve que ce qui a du sens pour un
lecteur.** Un champ vide est omis plutôt que présent à vide, `EntityContact`
disparaît, et le `NaceCode` brut est jeté une fois sa description résolue.

> **Pourquoi ne pas tout faire d'un coup en une seule couche ?** Parce que les
> deux étapes n'ont ni le même rythme ni le même risque. Le bronze est cher
> (~1 h de jointures) mais stable. Le silver est rapide et change souvent : une
> traduction à corriger, une règle métier qui évolue. Les séparer permet de
> rejouer le silver en quelques minutes **sans retoucher au bronze**. C'est tout
> l'intérêt de l'architecture en médaillon.

---

## Configuration

In [ ]:
import json
import os
import re
import time

import pymongo

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
DB_NAME = os.getenv("MONGO_DB", "kbo")
SOURCE = "entreprise"           # collection bronze (notebook precedent)
TARGET = "entreprise_silver"    # collection silver (produite ici)

client = pymongo.MongoClient(MONGO_URI, socketTimeoutMS=None)
db = client[DB_NAME]

def show(document) -> None:
    print(json.dumps(document, indent=2, ensure_ascii=False, default=str))

print("bronze :", SOURCE, f"({db[SOURCE].count_documents({}):,} documents)")
print("silver :", TARGET)

---

## 1. Lecture du bronze

Charger `entreprise` et `kbo_code` filtré sur `Language="FR"`.

### Le référentiel tient en mémoire : on en profite

`kbo_code` fait 21 468 lignes, dont **7 156 en français**. C'est minuscule. Plutôt
que d'ajouter un `$lookup` sur `kbo_code` pour chaque code à traduire — il y en a
plus de 40 millions à résoudre au total — on charge la table **une fois** dans un
dictionnaire Python indexé par `(Category, Code)`.

Chaque traduction devient alors un accès en O(1), sans aller-retour réseau. C'est
le pattern classique du *broadcast join* : quand un côté de la jointure est petit,
on le réplique en mémoire au lieu de le joindre.

Deux détails d'implémentation :

- **`(Category, Code)` en clé, pas `Code` seul.** Le code `001` existe dans
  `ActivityGroup` (« Activités TVA »), dans `TypeOfDenomination` (« Dénomination »)
  et dans `JuridicalSituation` (« Situation normale »). Sans la catégorie, on
  traduirait n'importe quoi en n'importe quoi.
- **`.strip()` sur la description.** Le code `TypeOfDenomination=004` vaut
  `" Dénomination de la succursale"`, avec une espace initiale dans la source.
  Sans nettoyage, elle deviendrait une clé de dictionnaire bancale.

In [ ]:
LANGUAGE = "FR"

def load_code_table(language: str = LANGUAGE) -> dict[tuple[str, str], str]:
    """`kbo_code` filtre sur une langue, indexe en memoire par (categorie, code)."""
    return {
        (doc["Category"], doc["Code"]): doc["Description"].strip()
        for doc in db.kbo_code.find(
            {"Language": language},
            {"_id": 0, "Category": 1, "Code": 1, "Description": 1},
        )
    }


CODES = load_code_table()

def translate(category: str, code, default=None):
    """Libelle francais d'un code ; `default` si le code est vide ou inconnu."""
    if not code:
        return default
    return CODES.get((category, code), default)


print(f"{len(CODES):,} codes charges en memoire\n")
for category, code in [("Status", "AC"), ("TypeOfAddress", "REGO"),
                       ("TypeOfDenomination", "001"), ("Language", "2"),
                       ("ActivityGroup", "006"), ("JuridicalForm", "014")]:
    print(f"  {category:<20} {code:<5} -> {translate(category, code)!r}")

print("\nle meme code, trois sens differents :")
for category in ("ActivityGroup", "TypeOfDenomination", "JuridicalSituation"):
    print(f"  ({category:<20}, '001') -> {translate(category, '001')!r}")

Le document bronze de départ, pour comparaison avec ce qui suit :

In [ ]:
REFERENCE = "0200.245.711"
bronze = db[SOURCE].find_one({"_id": REFERENCE})

print("champs plats :", {k: v for k, v in bronze.items()
                         if not isinstance(v, list)})
print("\ndenominations brutes :")
show(bronze["denominations"])

---

## 2. Champs scalaires codés

`Status`, `JuridicalSituation`, `TypeOfEnterprise`, `JuridicalForm`, `JuridicalFormCAC` : cinq champs plats, chacun traduit indépendamment via `kbo_code`. Un champ absent ou vide dans le bronze (ex. `JuridicalFormCAC=""`) doit être **omis** du silver plutôt que d'y apparaître traduit en valeur nulle.

Chaque champ est décrit par un triplet **(nom de sortie, champ bronze, catégorie
`kbo_code`)**. Deux subtilités :

- `JuridicalFormCAC` se traduit avec la catégorie **`JuridicalForm`** : c'est la
  même nomenclature de formes juridiques, dans un contexte différent. Le nom du
  champ et le nom de la catégorie ne coïncident donc pas toujours.
- L'omission est obtenue « gratuitement » par une compréhension de dictionnaire
  avec walrus : si `translate` renvoie `None` (code vide ou inconnu), la clé
  n'est simplement jamais créée. Pas de `if` imbriqué, pas de `None` résiduel.

Les champs sont déclarés dans l'ordre alphabétique de leur nom de sortie, ce qui
donne un ordre de clés stable et prévisible dans le document final.

In [ ]:
# (nom de sortie, champ bronze, categorie kbo_code)
SCALAR_FIELDS = (
    ("juridicalForm",      "JuridicalForm",      "JuridicalForm"),
    ("juridicalFormCAC",   "JuridicalFormCAC",   "JuridicalForm"),
    ("juridicalSituation", "JuridicalSituation", "JuridicalSituation"),
    ("status",             "Status",             "Status"),
    ("typeOfEnterprise",   "TypeOfEnterprise",   "TypeOfEnterprise"),
)

def clean_scalars(bronze: dict) -> dict:
    """Traduit les 5 champs plats ; un champ vide ou inconnu est omis."""
    return {
        name: label
        for name, field, category in SCALAR_FIELDS
        if (label := translate(category, bronze.get(field)))
    }


bronze = db[SOURCE].find_one({"_id": REFERENCE})
print("bronze :", {f: bronze.get(f) for _, f, _ in SCALAR_FIELDS})
print("\nsilver :")
show(clean_scalars(bronze))
print("\n-> JuridicalFormCAC valait \"\" : la cle est absente, et non traduite en null.")

Le document silver partiel, à ce stade :

In [ ]:
partial = {"_id": bronze["_id"], "enterpriseNumber": bronze["EnterpriseNumber"],
           "startDate": bronze["StartDate"], **clean_scalars(bronze)}
show(partial)

---

## 3. Dénominations : tableau → dict `{type traduit: {language, denomination}}`

Chaque entrée doit être keyée par son `TypeOfDenomination` **traduit**

### Pourquoi un dictionnaire plutôt qu'un tableau

Dans le bronze, pour lire l'abréviation d'une entreprise il faut **parcourir** le
tableau en testant `TypeOfDenomination == "002"`. Dans le silver, c'est
`doc["denominations"]["Abréviation"]`. La donnée n'a pas changé, sa
**structure d'accès** oui — et c'est tout l'intérêt d'une base documentaire.

Le type devient la clé ; il ne sert donc plus à rien de le répéter dans la valeur.
Il reste `{language, denomination}`, avec la langue elle aussi traduite
(`"2"` → `"néerlandais"`).

**Conflit de clés** : une entreprise peut porter deux dénominations du même type
(deux langues, par exemple). Un dictionnaire ne garde qu'une valeur par clé —
la spec tranche : *dernier gagne*. C'est le comportement naturel d'une
affectation en boucle, aucun code supplémentaire n'est nécessaire ; mais c'est
une **perte d'information assumée**, qu'il vaut mieux avoir décidée que subie.

In [ ]:
def clean_denominations(rows) -> dict:
    """Tableau -> dict {type traduit: {language, denomination}} ; dernier gagne."""
    result = {}
    for row in rows:
        label = translate("TypeOfDenomination", row.get("TypeOfDenomination"))
        if not label:
            continue
        result[label] = {
            "language": translate("Language", row.get("Language"), ""),
            "denomination": row.get("Denomination", ""),
        }
    return result


print("bronze :")
show(bronze["denominations"])
print("\nsilver :")
show(clean_denominations(bronze["denominations"]))

---

## 4. Adresses : tableau → dict `{type traduit: {country, street...}}`

Même principe clé-traduite que les dénominations, plus deux règles spécifiques :

- **`country`** : `CountryFR` nettoyé des mentions entre parenthèses (`"France (Métropole)"` → `"France"`) et des espaces multiples ; si le résultat est vide, mettre `"Belgique"` par défaut.
- **Champs vides omis** : `box`, `zipcode`... vides ne doivent pas apparaître dans le document silver.

### Trois décisions

**1. On garde le français, on jette le néerlandais.** Le bronze porte
`MunicipalityNL` *et* `MunicipalityFR`. La couche silver est francophone (comme
`kbo_code` filtré sur `FR`) : on ne conserve que les colonnes `*FR`. C'est un
choix éditorial, pas une perte — le bronze reste disponible pour produire une
variante néerlandophone.

**2. `country` vide signifie « Belgique ».** Dans l'export KBO, le pays n'est
renseigné **que pour les adresses étrangères**. Une valeur vide n'est donc pas une
donnée manquante : c'est une information implicite qu'on rend explicite. C'est
typiquement le genre de convention métier qu'une couche silver doit absorber pour
que le consommateur n'ait pas à la connaître.

**3. Les mentions entre parenthèses sont du bruit.** `"France (Métropole)"` et
`"France (Départements d'outre-mer)"` désignent le même pays pour qui veut
compter des entreprises par pays. On les retire, puis on normalise les espaces
laissés derrière.

> ⚠️ **Divergence assumée entre l'énoncé et les exemples.** La spec écrite demande
> d'omettre les champs vides (« `box`, `zipcode`... vides ne doivent pas
> apparaître »), mais **tous** les documents d'exemple de l'énoncé affichent
> `"box": ""`. Les deux sont incompatibles. J'implémente **la règle écrite**, en
> isolant l'arbitrage dans une constante — basculer d'un comportement à l'autre
> se fait en une ligne. Les deux sorties sont affichées ci-dessous.

In [ ]:
# La spec ecrite demande d'omettre les champs vides ; les exemples de l'enonce
# conservent `box: ""`. Constante = arbitrage explicite et reversible.
OMIT_EMPTY_ADDRESS_FIELDS = True

DEFAULT_COUNTRY = "Belgique"
_PARENTHESES = re.compile(r"\([^)]*\)")
_MULTI_SPACE = re.compile(r"\s+")

ADDRESS_FIELDS = (
    ("zipcode",      "Zipcode"),
    ("municipality", "MunicipalityFR"),
    ("street",       "StreetFR"),
    ("houseNumber",  "HouseNumber"),
    ("box",          "Box"),
)

def clean_country(raw: str | None) -> str:
    """'France (Metropole)' -> 'France' ; vide -> 'Belgique'."""
    country = _MULTI_SPACE.sub(" ", _PARENTHESES.sub(" ", raw or "")).strip()
    return country or DEFAULT_COUNTRY


for raw in ["France (Métropole)", "  Pays-Bas  ", "", None,
            "Royaume-Uni (Grande-Bretagne et Irlande du Nord)"]:
    print(f"  {raw!r:<52} -> {clean_country(raw)!r}")

In [ ]:
def clean_addresses(rows) -> dict:
    """Tableau -> dict {type traduit: {country, zipcode, municipality, ...}}."""
    result = {}
    for row in rows:
        label = translate("TypeOfAddress", row.get("TypeOfAddress"))
        if not label:
            continue
        address = {"country": clean_country(row.get("CountryFR"))}
        for name, field in ADDRESS_FIELDS:
            value = (row.get(field) or "").strip()
            if value or not OMIT_EMPTY_ADDRESS_FIELDS:
                address[name] = value
        result[label] = address
    return result


print("bronze :")
show(bronze["addresses"])
print("\nsilver (regle ecrite : champs vides omis) :")
show(clean_addresses(bronze["addresses"]))

OMIT_EMPTY_ADDRESS_FIELDS = False
print("\nsilver (variante des exemples de l'enonce : box conserve) :")
show(clean_addresses(bronze["addresses"]))
OMIT_EMPTY_ADDRESS_FIELDS = True   # on retablit la regle ecrite

Contrôle de la règle « pays vide → Belgique » sur une adresse réellement
étrangère (`0257.883.408`, une association turque) :

In [ ]:
foreign = db[SOURCE].find_one({"_id": "0257.883.408"})
print("CountryFR brut :", repr(foreign["addresses"][0]["CountryFR"]))
show(clean_addresses(foreign["addresses"]))

---

## 5. Contacts : tableau → dict `{email, phone, web}`

`EntityContact` (indique juste si le contact appartient à l'entreprise, un établissement ou une succursale) ne doit **jamais** être repris dans le silver. Un contact avec une valeur vide doit être ignoré.

### Le mapping ne peut pas venir de `kbo_code`

Réflexe naturel : traduire `ContactType` via `kbo_code` comme tout le reste. Ça
ne marche pas, pour **deux** raisons vérifiées sur les données :

1. `kbo_code` traduirait `EMAIL` en `"Adresse e-mail"` et `TEL` en
   `"Numéro de téléphone"` — or la sortie attendue veut des clés techniques
   courtes : `email`, `phone`, `web`.
2. Surtout : `contact.csv` contient **quatre** valeurs distinctes —
   `EMAIL`, `TEL`, `WEB` et **`FAX`** — alors que `kbo_code` n'en documente que
   **trois**. `FAX` n'y figure pas. Une traduction par `kbo_code` perdrait
   purement et simplement les numéros de fax.

D'où un mapping explicite en dur, complété d'un repli `.lower()` pour tout type
qui apparaîtrait dans une future livraison sans casser le pipeline.

**`EntityContact` est ignoré** : dans le bronze, un contact est déjà rattaché à
la bonne entité par construction. Savoir qu'il vaut `ENT` ou `EST` quand on le lit
depuis le document de l'entreprise n'apprend rien — c'est une redondance qu'on
supprime.

In [ ]:
CONTACT_KEYS = {"EMAIL": "email", "TEL": "phone", "WEB": "web", "FAX": "fax"}

def clean_contacts(rows) -> dict:
    """Tableau -> dict {email?, phone?, web?, fax?} ; `EntityContact` jamais lu."""
    result = {}
    for row in rows:
        value = (row.get("Value") or "").strip()
        if not value:                       # contact vide -> ignore
            continue
        contact_type = row.get("ContactType", "")
        result[CONTACT_KEYS.get(contact_type, contact_type.lower())] = value
    return result


print("types presents dans la source :", sorted(db.kbo_contact.distinct("ContactType")))
print("types documentes dans kbo_code:",
      sorted(d["Code"] for d in db.kbo_code.find({"Category": "ContactType", "Language": "FR"})))
print("-> FAX absent du referentiel : le mapping doit etre explicite.\n")

contact_demo = db[SOURCE].find_one({"_id": "0201.543.234"})
print("bronze :")
show(contact_demo["contacts"])
print("\nsilver :")
show(clean_contacts(contact_demo["contacts"]))

---

## 6. Activités : dédoublonnage inter-versions NACE + répartition main/secondary

Le point le plus subtil de toute la couche silver. Une même activité réelle est souvent codée sous **plusieurs versions NACE** (2003, 2008, 2025) avec des libellés différents mais qui décrivent la même chose. Règle : dédoublonner sur `(activityGroup, description)` et, en cas de collision, **garder la version NACE la plus récente**. Le `NaceCode` brut ne doit **jamais** être gardé dans le silver, une fois `description` résolue via `Nace{version}`, le code numérique ne sert plus à un lecteur humain. Répartir le résultat en `{main: [...], secondary: [...]}` selon `Classification`.

### Pourquoi le code seul ne suffit jamais

La NACE est une nomenclature d'activités **révisée périodiquement**. Le même code
numérique ne désigne pas la même chose d'une version à l'autre :

| Code | `Nace2008` | `Nace2025` |
|---|---|---|
| `35130` | Distribution d'électricité | **Transport** d'électricité |

Traduire un `NaceCode` sans regarder son `NaceVersion` produit donc des libellés
**faux**. D'où la résolution dans la catégorie `Nace{version}` — et d'où le fait
que le code brut, une fois la description obtenue, n'a plus aucune valeur pour un
lecteur : on le jette.

### Où porte exactement le dédoublonnage

L'énoncé dit « dédoublonner sur `(activityGroup, description)` ». Les données
imposent une précision. Dans le résultat attendu pour FLUVIUS, la paire
`("Activités TVA", "Production d'électricité")` apparaît **deux fois** :

- dans `main`, en version **2003** ;
- dans `secondary`, en version **2008**.

Or `Nace2003 40110` et `Nace2008 35110` renvoient la **même chaîne exacte**
`"Production d'électricité"` (vérifié ci-dessous). Si le dédoublonnage était
global, l'entrée 2008 aurait écrasé l'entrée 2003 et une seule survivrait.

→ **Le dédoublonnage s'applique à l'intérieur de chaque bucket `main` /
`secondary`**, pas sur l'ensemble. La clé effective est
`(classification, activityGroup, description)`.

### Et pourquoi on ne normalise surtout pas les apostrophes

Toujours chez FLUVIUS, `main` contient ces deux entrées :

- `"Distribution d'électricité"` (2008) — apostrophe droite `U+0027`
- `"Distribution d’électricité"` (2025) — apostrophe courbe `U+2019`

Ce sont deux chaînes **différentes**, donc deux entrées conservées. La comparaison
se fait sur la chaîne brute, sans normalisation Unicode : la NACE 2025 a changé de
convention typographique, et « corriger » les apostrophes ferait fusionner ces
entrées et divergerait du résultat attendu.

In [ ]:
# La demonstration des deux points ci-dessus, sur le referentiel reel.
print("Meme code, deux versions, deux sens :")
for version in ("2008", "2025"):
    print(f"  Nace{version} 35130 -> {translate(f'Nace{version}', '35130')!r}")

print("\nDeux codes differents, la MEME chaine exacte :")
for version, nace in (("2003", "40110"), ("2008", "35110")):
    print(f"  Nace{version} {nace} -> {translate(f'Nace{version}', nace)!r}")
print("  identiques :", translate("Nace2003", "40110") == translate("Nace2008", "35110"))

print("\nConvention typographique par version :")
for version in ("2003", "2008", "2025"):
    droite = sum(1 for (cat, _), desc in CODES.items()
                 if cat == f"Nace{version}" and "\u0027" in desc)
    courbe = sum(1 for (cat, _), desc in CODES.items()
                 if cat == f"Nace{version}" and "\u2019" in desc)
    print(f"  Nace{version} : apostrophe droite {droite:>5}   apostrophe courbe {courbe:>5}")

In [ ]:
def clean_activities(rows) -> dict:
    """Dedoublonne les activites et les repartit en {main, secondary}.

    - la description est resolue dans la categorie `Nace{version}` : un meme
      code ne veut pas dire la meme chose d'une version a l'autre ;
    - dedoublonnage sur (activityGroup, description) *a l'interieur de chaque
      classification*, la version NACE la plus recente gagne ;
    - le `NaceCode` brut n'est jamais conserve.
    """
    buckets: dict[str, dict] = {"main": {}, "secondary": {}}

    for row in rows:
        version = row.get("NaceVersion", "")
        description = translate(f"Nace{version}", row.get("NaceCode"))
        if not description:                 # code inconnu du referentiel
            continue

        group = translate("ActivityGroup", row.get("ActivityGroup"), "")
        bucket = buckets["main" if row.get("Classification") == "MAIN" else "secondary"]
        key = (group, description)

        previous = bucket.get(key)
        if previous is None or version > previous["naceVersion"]:
            bucket[key] = {"activityGroup": group,
                           "description": description,
                           "naceVersion": version}

    return {name: list(entries.values()) for name, entries in buckets.items()}

Sur FLUVIUS (`0201.311.226`), 11 activités brutes se réduisent à 11 entrées
lisibles : le dédoublonnage n'écrase que ce qui décrit réellement la même chose.

In [ ]:
fluvius = db[SOURCE].find_one({"_id": "0201.311.226"})

print(f"activites brutes : {len(fluvius['activities'])}")
for row in fluvius["activities"][:6]:
    print("   ", {k: v for k, v in row.items() if k != "_id"})
print("    ...")

result = clean_activities(fluvius["activities"])
print(f"\nsilver : {len(result['main'])} main + {len(result['secondary'])} secondary")
show(result)

> **Note sur `Classification`.** La nomenclature prévoit trois valeurs : `MAIN`,
> `SECO` (secondaire) et `ANCI` (auxiliaire). L'énoncé n'impose que deux buckets ;
> tout ce qui n'est pas `MAIN` part donc dans `secondary`, ce qui évite de perdre
> silencieusement les activités auxiliaires.

---

## 7. Établissements : mêmes règles, en plus léger

Un établissement a ses propres dénominations/adresses/contacts/activités, à nettoyer avec **exactement les mêmes règles** que ci-dessus. Seule différence avec le document entreprise : pas d'`EnterpriseNumber` (déjà le document de cette entreprise, ce serait une redondance pure). Résultat keyé par `EstablishmentNumber`.

« Exactement les mêmes règles » se traduit littéralement en code : on **réutilise
les quatre fonctions déjà écrites**, sans les dupliquer ni les paramétrer. C'est
le bénéfice d'avoir écrit des fonctions qui prennent un *tableau de lignes* en
entrée plutôt qu'un document entreprise entier.

Le numéro d'établissement passe du corps du document vers la **clé** du
dictionnaire — comme les types de dénomination à la section 3. Et
`EnterpriseNumber` disparaît : on est déjà dans le document de cette entreprise,
le répéter à chaque établissement serait une redondance pure.

In [ ]:
def clean_establishment(row: dict) -> dict:
    """Memes regles que l'entreprise, sans `EnterpriseNumber` (redondant ici)."""
    document = {}
    if row.get("StartDate"):
        document["startDate"] = row["StartDate"]
    document["denominations"] = clean_denominations(row.get("denominations", ()))
    document["addresses"] = clean_addresses(row.get("addresses", ()))
    document["contacts"] = clean_contacts(row.get("contacts", ()))
    document["activities"] = clean_activities(row.get("activities", ()))
    return document


establishments = {row["EstablishmentNumber"]: clean_establishment(row)
                  for row in fluvius["establishments"]}
show(establishments)

---

## 8. Succursales : encore plus léger

Une succursale n'a jamais de dénomination propre ni d'activité propre : ne nettoyer que `addresses` et `contacts`. Ni `EnterpriseNumber`, ni `denominations`, ni `activities` dans la sortie. Résultat keyé par `Id`.

Le bronze produit bien `denominations: []` et `activities: []` pour les
succursales — la fonction `_detail_lookups` du premier notebook applique les
quatre jointures uniformément. Le silver **supprime ces deux clés** au lieu de les
laisser vides : une clé toujours vide est du bruit, et sa présence laisserait
croire qu'une succursale *pourrait* en avoir.

C'est exactement la division du travail entre les deux couches : le bronze reste
uniforme et mécanique, le silver applique la connaissance métier.

In [ ]:
def clean_branch(row: dict) -> dict:
    """Une succursale n'a jamais de denomination ni d'activite propre."""
    document = {}
    if row.get("StartDate"):
        document["startDate"] = row["StartDate"]
    document["addresses"] = clean_addresses(row.get("addresses", ()))
    document["contacts"] = clean_contacts(row.get("contacts", ()))
    return document


itkib = db[SOURCE].find_one({"_id": "0257.883.408"})
print("bronze (cles vides conservees par le pipeline de jointure) :")
show({k: v for k, v in itkib["branches"][0].items() if k != "_id"})
print("\nsilver :")
show({row["Id"]: clean_branch(row) for row in itkib["branches"]})

---

## Assemblage : le document silver complet

`to_silver` ne fait qu'ordonner les briques précédentes. L'ordre des clés est
volontaire — identifiants, puis les blocs imbriqués du plus général au plus
spécifique, puis les champs scalaires traduits (alphabétiques) en fin de
document, là où ils ne gênent pas la lecture.

C'est une **fonction pure** : elle lit un document bronze et renvoie un nouveau
document, sans jamais modifier son entrée. Elle est donc testable unitairement,
sans base de données, et rejouable sans effet de bord.

In [ ]:
def to_silver(bronze: dict) -> dict:
    """Document bronze -> document silver (fonction pure)."""
    document = {
        "_id": bronze["_id"],
        "enterpriseNumber": bronze.get("EnterpriseNumber") or bronze["_id"],
    }
    if bronze.get("StartDate"):
        document["startDate"] = bronze["StartDate"]

    document["denominations"] = clean_denominations(bronze.get("denominations", ()))
    document["addresses"] = clean_addresses(bronze.get("addresses", ()))
    document["contacts"] = clean_contacts(bronze.get("contacts", ()))
    document["activities"] = clean_activities(bronze.get("activities", ()))
    document["establishments"] = {
        row["EstablishmentNumber"]: clean_establishment(row)
        for row in bronze.get("establishments", ())
    }
    document["branches"] = {row["Id"]: clean_branch(row)
                            for row in bronze.get("branches", ())}

    document.update(clean_scalars(bronze))
    return document


show(to_silver(db[SOURCE].find_one({"_id": "0257.883.408"})))

---

## 9. Écriture dans `entreprise_silver`

Snapshot complet : vider `entreprise_silver` puis écrire le résultat.

La transformation se fait **en streaming** : un curseur parcourt le bronze,
`to_silver` s'applique document par document, et les résultats repartent par
paquets de 2 000. À aucun moment plus de quelques milliers de documents ne
tiennent en mémoire — les 1,95 million ne sont jamais chargés d'un bloc.

`batch_size=200` sur le curseur limite la taille des lots renvoyés par le serveur :
les documents bronze sont volumineux (jusqu'à 3 Mo), un lot par défaut de 101
documents « gros » ferait gonfler la mémoire côté client sans bénéfice.

Le `drop()` initial fait de l'opération un **snapshot complet** : le résultat ne
dépend pas de l'état antérieur de la collection, donc rejouer la cellule donne
toujours le même résultat.

In [ ]:
def build_silver(batch_size: int = 2_000) -> int:
    """Snapshot complet : on vide la cible puis on la reconstruit en streaming."""
    db[TARGET].drop()
    written, buffer, started = 0, [], time.perf_counter()

    for bronze_document in db[SOURCE].find(batch_size=200):
        buffer.append(to_silver(bronze_document))
        if len(buffer) >= batch_size:
            db[TARGET].insert_many(buffer, ordered=False)
            written += len(buffer)
            buffer.clear()

    if buffer:
        db[TARGET].insert_many(buffer, ordered=False)
        written += len(buffer)

    elapsed = time.perf_counter() - started
    print(f"{TARGET} : {written:,} documents en {elapsed / 60:.1f} min "
          f"({written / elapsed:,.0f}/s)")
    return written


build_silver()

In [ ]:
bronze_stats = db.command("collStats", SOURCE, scale=1024 * 1024)
silver_stats = db.command("collStats", TARGET, scale=1024 * 1024)

print(f"{'':<12}{'documents':>12}{'donnees':>12}{'doc moyen':>12}")
for label, stats in (("bronze", bronze_stats), ("silver", silver_stats)):
    print(f"{label:<12}{stats['count']:>12,}{stats['size']:>11,.0f}M"
          f"{stats['avgObjSize'] / 1024:>11,.1f}K")

gain = 1 - silver_stats["size"] / bronze_stats["size"]
print(f"\n-> le silver est {gain:.0%} plus compact que le bronze "
      f"(codes resolus, doublons NACE et champs vides supprimes)")

---

## 10. Vérification

Comparer un document silver produit à ce que prédit la spec ci-dessus, sur une entreprise connue.

On vérifie sur `0201.105.843` (I.D.E.A., une intercommunale de Mons avec 24
établissements), puis on transforme **chaque ligne du schéma cible de la
section 12 en assertion automatique** exécutée sur un échantillon large. Une
inspection visuelle ne prouve rien sur 1,95 million de documents.

In [ ]:
show(db[TARGET].find_one({"_id": "0201.105.843"}))

### Contrôles automatiques du contrat de la section 12

In [ ]:
SAMPLE_SIZE = 20_000
ADDRESS_KEYS = {"country", "zipcode", "municipality", "street", "houseNumber", "box"}
ACTIVITY_KEYS = {"activityGroup", "description", "naceVersion"}

checks: dict[str, int] = {}

def fails(name: str) -> None:
    checks[name] = checks.get(name, 0) + 1

for document in db[TARGET].aggregate([{"$sample": {"size": SAMPLE_SIZE}}]):
    if not isinstance(document["_id"], str):
        fails("_id doit etre une chaine")
    if document.get("enterpriseNumber") != document["_id"]:
        fails("enterpriseNumber == _id")

    for label, entry in document["denominations"].items():
        if label.strip() != label:
            fails("cle de denomination non nettoyee")
        if set(entry) - {"language", "denomination"}:
            fails("denomination : cle inattendue")

    for entry in document["addresses"].values():
        if not entry.get("country"):
            fails("country toujours renseigne")
        if set(entry) - ADDRESS_KEYS:
            fails("adresse : cle inattendue")
        if any(value == "" for value in entry.values()):
            fails("adresse : champ vide non omis")

    if any(value == "" for value in document["contacts"].values()):
        fails("contact vide non ignore")
    if "EntityContact" in document["contacts"]:
        fails("EntityContact present")

    for bucket in ("main", "secondary"):
        seen = set()
        for entry in document["activities"][bucket]:
            if set(entry) != ACTIVITY_KEYS:
                fails("activite : cles inattendues (naceCode ?)")
            key = (entry["activityGroup"], entry["description"])
            if key in seen:
                fails(f"doublon (activityGroup, description) dans {bucket}")
            seen.add(key)

    for number, establishment in document["establishments"].items():
        if not number.startswith("2."):
            fails("etablissement : cle != EstablishmentNumber")
        if "EnterpriseNumber" in establishment:
            fails("etablissement : EnterpriseNumber redondant")

    for number, branch in document["branches"].items():
        if not number.startswith("9."):
            fails("succursale : cle != Id")
        if set(branch) - {"startDate", "addresses", "contacts"}:
            fails("succursale : denominations/activities non supprimees")

print(f"{SAMPLE_SIZE:,} documents verifies\n")
if checks:
    for name, count in sorted(checks.items()):
        print(f"  ECHEC  {name} ({count})")
else:
    print("  Toutes les regles du schema cible sont respectees.")

### Le même document, avant et après

In [ ]:
before = db[SOURCE].find_one({"_id": "0200.245.711"})
after = db[TARGET].find_one({"_id": "0200.245.711"})

print("BRONZE — adresse brute")
show({k: v for k, v in before["addresses"][0].items() if k != "_id"})
print("\nSILVER — la meme adresse")
show(after["addresses"])

print("\nBRONZE — champs plats codes")
show({k: v for k, v in before.items() if not isinstance(v, list) and k != "_id"})
print("\nSILVER — les memes champs, traduits")
show({k: v for k, v in after.items() if isinstance(v, str) and k != "_id"})

---

## 12. Schéma cible de `entreprise_silver`

| Champ | Type | Origine / règle |
|---|---|---|
| `_id` | string | copié tel quel du bronze |
| `enterpriseNumber` | string | `EnterpriseNumber` (ou `_id` en secours) |
| `startDate` | string, optionnel | copié si présent |
| `status`, `juridicalSituation`, `typeOfEnterprise`, `juridicalForm`, `juridicalFormCAC` | string, optionnels | traduits FR via `code.csv`, omis si absents du bronze |
| `denominations` | dict `{type traduit: {language, denomination}}` | dernier gagne en cas de type dupliqué |
| `addresses` | dict `{type traduit: {country, zipcode, municipality, street, houseNumber, box}}` | champs vides omis, `country` nettoyé + défaut `"Belgique"` |
| `contacts` | dict `{email?, phone?, web?}` | `EntityContact` jamais lu |
| `activities` | `{main: [...], secondary: [...]}` | dédoublonné par `(activityGroup, description)`, version NACE la plus récente gagne, `naceCode` brut jamais gardé |
| `establishments` | dict `{EstablishmentNumber: {startDate?, denominations, addresses, contacts, activities}}` | mêmes règles que l'entreprise, sans `EnterpriseNumber` |
| `branches` | dict `{Id: {startDate?, addresses, contacts}}` | sans `denominations` ni `activities` (une succursale n'en a jamais) |

---

## Bilan

`entreprise_silver` est directement exploitable : plus un seul code à décoder,
plus un seul tableau à parcourir pour trouver une information, plus une seule
activité en double.

Les trois points qui demandaient une vraie lecture des données — et non
seulement de l'énoncé :

1. **`FAX` existe dans `contact.csv` mais pas dans `code.csv`** → le mapping des
   types de contact doit être explicite, sinon on perd des données.
2. **Un code NACE change de sens d'une version à l'autre** (`35130` = distribution
   en 2008, transport en 2025) → la description doit être résolue dans
   `Nace{version}`, jamais dans une table unique.
3. **Le dédoublonnage porte sur chaque bucket, pas sur l'ensemble** → prouvé par
   `("Activités TVA", "Production d'électricité")` présent à la fois en `main`
   (2003) et en `secondary` (2008), les deux versions renvoyant la même chaîne.

Une divergence entre l'énoncé et ses exemples a été relevée et arbitrée
explicitement (section 4, `OMIT_EMPTY_ADDRESS_FIELDS`).

**Suite logique — la couche gold.** Le silver reste un miroir fidèle du registre,
document par document. Une couche gold agrégerait pour répondre à des questions
métier : nombre d'entreprises actives par commune et par secteur, séries
temporelles de créations, tables dénormalisées prêtes pour un outil de BI.